**Memisahkan Dataset**

In [ ]:
import pandas as pd

# Membaca dataset
df = pd.read_csv("data_manual.csv")

# Memisahkan data berlabel dan tidak berlabel
data_labeled = df[df['Label'].notna()].copy()
data_unlabeled = df[df['Label'].isna()].copy()

print("Jumlah data berlabel :", len(data_labeled))
print("Jumlah data tidak berlabel :", len(data_unlabeled))

# Menyimpan data berlabel ke file baru
data_labeled.to_csv("data_labeled.csv", index=False)


**Preprocessing**

In [ ]:
pip install Sastrawi nltk

In [ ]:
pip install emoji

In [ ]:
import nltk
nltk.download("stopwords")

In [ ]:
import re
import pandas as pd
import emoji
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

# 1. LOAD CSV
df = pd.read_csv("data_labeled.csv")

# STEMMER
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# pastikan kolom teks ada
print(df.columns)                 # cek kolom

# STOPWORDS (tidak hapus kata penting)
stopwords_custom = set([
    "nya","mbak","devita","ya","mba","mas","loh","deh","t","di","d","ala","h","lohh","b","fifin","celsi","silvia","dll","okay","lena","agam","salma",
    "rosvina","m","yak","hapir","n","g","e","edge","iyain","thania","dng","daan","yang","untuk","dan","di","ke","dari","ini","itu","juga","kami","kamu","saya","meirizka",
    "ada","dengan","pada","karena","agar","atau","jadi","tidak","iya","yg","nya","ya","tapi","powwl","ny","nih","lahh","zxz","buat","dalam","para","akan","sudah","serta","sih","hehe","hmm","deh","mbak","mba","pool","tania",
    "mas","kak","caroline","devita","lena","dg","fifin","tp","dgn","loh","t","b","aja","imo","amoun","saja","pun","itu","anda","agar","yakni","sebagai","maka","yaitu","silvia","ad","sy","san","kpd","the","untvlantai","lo",
    "bs","d","n","sll","aztuti","dkt", "tx", "u", "anak","aku","devita","best","dj","thania","fres","asa","dll","yak","hahaha","ken","siip","kita","madam","diksh","kecil","h","yahh","x","mb","st","fo","silviaa","ka",
    "f","b","sm","tdk","nur","si","kap","m","yuuk","parpel","chelsi","dg","sgt,","tpi","se","occ","daan","an","andin","traine","trainee","s","mau","kes","asa","jd","ku","dik","welcome","dr","thania"
])


# 2. Normalisasi
normalisasi_map = {
    "byr": "bayar","hrus": "harus","but": "tapi","tpi": "tapi","jan": "januari","diprepare": "prepare","aps": "aplikasi","kyk": "seperti","pool": "banget","dikurangi": "kurang",
    "tlp": "telepon","feb": "februari","dlm": "dalam","sxan": "sekalian","dng": "dengan","budhenya": "bude","dikususkan": "khusus","bervariasiiii": "bervariasi","servicenya": "service",
    "raman": "ramah","beragam": "ragam","diningnya": "dining","petigas": "petugas","nutupnya": "tutup", "mmg": "memang","lwt": "lewat","belain": "bela",
    "wa": "whatsapp","sarapanya": "sarapan", "th": "tahun", "tau": "mengerti", "dimsumnya": "dimsum","rb": "ribu", "biryaninya": "briyani", "orng": "orang","klw": "jika","sxan": "sekalian",
    "terallu": "terlalu","amoun": "ampun","bgtt":"banget","bgt":"banget","bangett":"banget","byk":"banyak","inn":"in","bnget":"banget","kren":"keren","suuperr":"bagus","servicenha":"servis",
    "gpp":"gapapa","ga":"tidak","gk":"tidak","nggak":"tidak","ngga":"tidak","kram":"kran","dkt":"dekat","cek":"check","pesen":"pesan",
    "makasih":"terima kasih","mksh":"terima kasih","thx":"terima kasih","jl":"jalan","dtg":"datang","baguss":"bagus","thank you":"terimakasih",
    "oke":"baik","ok":"baik","sip":"baik","bberapa":"beberapa","asa":"aja","lg":"lagi","msk":"masuk","tv":"televisi","clear":"bersih","tsb":"tersebut","overalla":"over all",
    "rekomen":"rekomendasi","recommended":"rekomendasi","sangatt":"sangat","maknyus":"enak","topp":"bagus","thank u":"terimaksih","tengkyu":"terimakasih",
    "jg":"juga","jd":"jadi","dpt":"dapat","dapet":"dapat","apalgii":"apalagi","restorannlt":"restoran","desertnya":"desert","trmmksh":"terimakasih",
    "km":"kamu","tmn":"teman","tmpt":"tempat","tks":"terimakasih","berasa":"rasa","ksni":"kesini","sus":"aneh","baguus":"bagus","besty":"best","trimakasih":"terimakasih",
    "udh":"sudah","sdh":"sudah","blm":"belum","hr":"hari","wlopun":"walaupun","happy":"senang","bussines trip":"perjalanan bisnis","ga":"tidak",
    "kl":"kalau","klo":"kalau","kalo":"kalau","tksh":"terimakasih","cantikk":"cantik","nggk":"tidak","menservice":"servis","overall":"semua","sat set":"cepat",
    "krn":"karena","trm":"terima","trs":"terus","bgs":"bagus","breakfastnya":"breakfast","enakk":"enak","thank":"terimakasih","lt":"lantai",
    "bbrp":"beberapa","kmr":"kamar","dgn":"dengan","lbh":"lebih","enakk":"enak","baguss":"bagus","trus":"terus","gak":"tidak","bnyk":"banyak","amburadul":"berantakan",
    "mantappp":"mantap","mantapppp":"mantap","mantappppp":"mantap","sukakk":"suka","lonsay":"lontong sayur","too bad":"buruk","deal":"setuju","tgl":"tanggal","agam":"agak","kg":"juga",
    "parahh":"parah","pdhal":"padahal","tpat":"tempat","lg":"lagi","yogjakarta":"yogyakarta","pelayannann":"pelayanan","recomend":"recomended",
    "thank u":"terima kasih","utk":"untuk","org":"orang","pengkap":"lengkap","rs":"rumah sakit","breakfast":"sarapan","pdhl":"padahal","bukber":"buka bersama","hotell":"hotel",
    "bgt": "banget", "bget": "banget","bener": "benar", "bener2": "benar benar","udh": "sudah", "udah": "sudah","tdk": "tidak", "tida": "tidak", "td": "tidak",
    "dgn": "dengan", "dg": "dengan","dr": "dari","sm": "sama","tp": "tapi","yg": "yang","klo": "kalau", "kl": "kalau","krn": "karena", "karna": "karena","dpt": "dapat","bsk": "besok","bs": "bisa",
    "hr": "hari","cm": "cuma","aja": "saja","sy": "saya", "sya": "saya","km": "kamu", "kmu": "kamu","dmn": "di mana","knp": "kenapa","bgt": "banget","bgtt": "banget","jd": "jadi","gmn": "bagaimana",
    "sm": "sama","nglakuin": "melakukan", "ngelakuin": "melakukan","ngerasa": "merasa","nunggu": "menunggu","ngirim": "mengirim","ngasih": "memberi","ngambil": "mengambil","wkwk": "",
    "lol": "","btw": "omong-omong","ty": "terima kasih", "thx": "terima kasih","2x": "dua kali","1x": "sekali","gk": "tidak","ga": "tidak","ngga": "tidak","nggak": "tidak","tdk": "tidak","tbh": "tambah",
    "tp": "tapi","jg": "juga","jd": "jadi","dgn": "dengan","sm": "sama","sy": "saya","aq": "aku","gw": "saya","gue": "saya","km": "kamu","kmu": "kamu","dr": "dari","krn": "karena","pls": "tolong","plis": "tolong","bgt": "banget","bngt": "banget",
    "udh": "sudah","sdh": "sudah","dl": "dulu","blm": "belum","brp": "berapa","ttp": "tetap","ok": "baik","oke": "baik","nya": "", "mbak": "", "mba": "", "mas": "", "loh": "", "lohh": "","ya": "", "t": "", "di": "", "d": "", "ala": "", "h": "", "b": "",
    "fifin": "", "celsi": "", "silvia": "", "okay": "", "lena": "","agam": "", "salma": "", "rosvina": "", "m": "", "yak": "","hapir": "hampir", "n": "", "g": "", "e": "", "edge": "", "iyain": "iya", "thania": "", "dng": "dengan", "daan": "",
    "nya": "","mbak": "","devita": "","ya": "","mba": "","mas": "","loh": "","deh": "","t": "","di": "","d": "","ala": "","h": "","lohh": "","b": "","fifin": "","celsi": "","silvia": "","dll": "","okay": "","lena": "","agam": "","salma": "",
    "rosvina": "","m": "","yak": "","hapir": "","n": "","g": "","e": "","edge": "","iyain": "","thania": "","dng": "","daan": "","yang": "","untuk": "","dan": "","di": "","ke": "","dari": "","ini": "","itu": "","juga": "","kami": "","kamu": "","saya": "","meirizka": "",
    "ada": "","dengan": "","pada": "","karena": "","agar": "","atau": "","jadi": "","tidak": "","iya": "","yg": "","nya": "","ya": "","tapi": "","powwl": "","ny": "","nih": "","lahh": "","zxz": "",
    "buat": "","dalam": "","para": "","akan": "","sudah": "","serta": "","sih": "","hehe": "","hmm": "","deh": "","mbak": "","mba": "","pool": "","tania": "",
    "mas": "","kak": "","caroline": "","devita": "","lena": "","dg": "","fifin": "","tp": "","dgn": "","loh": "","t": "","b": "","aja": "","imo": "","amoun": "",
    "saja": "","pun": "","itu": "","anda": "","agar": "","yakni": "","sebagai": "","maka": "","yaitu": "","silvia": "","ad": "","sy": "","san": "","kpd": "","the": "","untvlantai": "","lo": "",
    "bs": "","d": "","n": "","sll": "","aztuti": "","dkt": "", "tx": "", "u": "", "anak": "","aku": "","devita": "","best": "","dj": "","thania": "","fres": "","asa": "",
    "dll": "","yak": "","hahaha": "","ken": "","siip": "","kita": "","madam": "","diksh": "","kecil": "","h": "","yahh": "","x": "","mb": "","st": "","fo": "","silviaa": "","ka": "",
    "f": "","b": "","sm": "","nur": "","si": "","kap": "","m": "","yuuk": "","parpel": "","chelsi": "","dg": "","sgt": "","tpi": "","se": "","occ": "","daan": "",
    "an": "","andin": "","traine": "","trainee": "","s": "","mau": "","kes": "","asa": "","jd": "","ku": "","dik": "","welcome": "","dr": "","thania": "",
    "istrht": "istirahat", "berteletele": "", "smua": "semua", "bambang": "", "diupgrade": "upgrade", "moment": "momen", "klau": "jika", "malem": "malam", "kenceng": "cepat", "staffnya": "staff",
    "makanannnya": "makan", "banquet": "banget", "lgs": "langsung", "gara": "karena", "ngadain": "ada", "soso": "", "bobrok": "rusak", "engga": "tidak", "managementnya": "management", "pas": "", "gt": "",
    "gaada": "tidak ada", "antre": "antri", "pencrt": "pencet", "tanggungjawabnya": "tanggung jawab", "was": "", "mgkn": "mungkin", "sblmnya": "sebelum", "spreinya": "sprei", "kalaublita": "jika",
    "drink": "minum", "by": "lewat", "freon": "", "pas": "", "mlm": "malam", "tifak": "tidak", "spertinya": "seperti", "tmtp": "tempat",
    "waiternya": "pelayan", "pelayanya": "pelayan", "impressionnya": "kesan", "ngambilin": "ambil", " wi fi": "wifi", "roomnya": "room", "disetting": "atur", "spt": "", "buanyaak": "banyak"
}

# 3. Fungsi Normalisasi
def normalize_text(text):
    words = text.split()
    new_words = []
    for w in words:
        if w in normalisasi_map: # Use the renamed dictionary
            new_words.append(normalisasi_map[w])
        else:
            new_words.append(w)
    return " ".join(new_words)


# 4. Inisiasi stopword & stemmer
stop_words = set(stopwords.words("indonesian"))
factory = StemmerFactory()
stemmer = factory.create_stemmer()


# 5. Fungsi Preprocessing Lengkap
# =========================
def clean_text(text):
    text = str(text).lower() # Ensure text is string and handle NaN by converting to 'nan' then cleaning
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)     # hapus URL
    text = re.sub(r"\S+@\S+\.\S+", " ", text)               # hapus email
    text = emoji.replace_emoji(text, replace=' ')           # hapus emoji
    text = re.sub(r"[^a-zA-Z\s]", " ", text)                # hapus simbol & angka
    text = re.sub(r"\s+", " ", text).strip()                # rapikan spasi

    text = normalize_text(text)                             # normalisasi singkatan

    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]     # stopwords

    stems = [stemmer.stem(w) for w in tokens]               # stemming

    return " ".join(stems)

# 6. Terapkan ke CSV
df["Clean_Text"] = df["Casefold"].apply(clean_text)

# simpan hasil
df.to_csv("data_clean.csv", index=False)


Index(['Label', 'Casefold'], dtype='object')


In [ ]:
df.head()

,Label,Casefold,Clean_Text
0,positif,pelayanannya ramah bgtt fasilitas kamar juga o...,layan ramah banget fasilitas kamar rekomendasi...
1,negatif,baru kali ini sangat kecewa dengan pelayanan h...,kali kecewa layan hotel mercure kota kali kece...
2,positif,liburan menyenangkan di jogja stay di hotel yg...,libur senang jogja stay hotel
3,positif,lokasinya juga mantap dekat dengan bandara dan...,lokasi mantap bandara wisata kamar nyaman fasi...
4,positif,saya merekomendasikan anda untuk menginap di h...,rekomendasi inap hotel layan bagus ramah makan...


**Stemming/Lematisasi**

In [ ]:
# STEMMING
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
factory = StemmerFactory()
stemmer = factory.create_stemmer()
df['stem'] = df['Clean_Text'].apply(lambda x: stemmer.stem(x))

In [ ]:
df.head()

,Label,Casefold,Clean_Text,stem
0,positif,pelayanannya ramah bgtt fasilitas kamar juga o...,layan ramah banget fasilitas kamar rekomendasi...,layan ramah banget fasilitas kamar rekomendasi...
1,negatif,baru kali ini sangat kecewa dengan pelayanan h...,kali kecewa layan hotel mercure kota kali kece...,kali kecewa layan hotel mercure kota kali kece...
2,positif,liburan menyenangkan di jogja stay di hotel yg...,libur senang jogja stay hotel,libur senang jogja stay hotel
3,positif,lokasinya juga mantap dekat dengan bandara dan...,lokasi mantap bandara wisata kamar nyaman fasi...,lokasi mantap bandara wisata kamar nyaman fasi...
4,positif,saya merekomendasikan anda untuk menginap di h...,rekomendasi inap hotel layan bagus ramah makan...,rekomendasi inap hotel layan bagus ramah makan...


In [ ]:
import re
import requests
import pandas as pd

url = "https://raw.githubusercontent.com/datascienceid/wordlist/master/kata-dasar.txt"
kamus = requests.get(url).text.split("\n")
kamus = set([k.strip() for k in kamus if k.strip() != ""])

sfx_list = ["kan", "annya", "nya", "an", "i"]
pfx_list = ["meng", "meny", "men", "mem", "me", "peng", "peny", "pen", "pem", "di", "ke", "se", "ber", "ter", "per"]

def smart_lemma_word(w):
    original = w

    # 1. langsung cocok di kamus
    if w in kamus:
        return w

    # 2. sufiks dulu
    for suf in sfx_list:
        if w.endswith(suf):
            base = w[:-len(suf)]
            if base in kamus:
                return base

    # 3. prefiks
    for pre in pfx_list:
        if w.startswith(pre):
            base = w[len(pre):]
            if base in kamus:
                return base

    # 4. kombinasi prefiks + sufiks
    for pre in pfx_list:
        if w.startswith(pre):
            temp = w[len(pre):]
            for suf in sfx_list:
                if temp.endswith(suf):
                    base = temp[:-len(suf)]
                    if base in kamus:
                        return base

    # 5. fallback paling aman → kembalikan kata utuh (tidak dipotong berlebihan seperti "lay", "lokas", dll)
    return original

def smart_lemma(sentence):
    if pd.isna(sentence):
        return ""
    return " ".join([smart_lemma_word(w) for w in sentence.split()])

df["lemma"] = df["stem"].apply(smart_lemma)

#Simpan File
df.to_csv("data_stem_lemma.csv", index=False)

In [ ]:
df.head()

,Label,Casefold,Clean_Text,stem,lemma
0,positif,pelayanannya ramah bgtt fasilitas kamar juga o...,layan ramah banget fasilitas kamar rekomendasi...,layan ramah banget fasilitas kamar rekomendasi...,layan ramah banget fasilitas kamar rekomendasi...
1,negatif,baru kali ini sangat kecewa dengan pelayanan h...,kali kecewa layan hotel mercure kota kali kece...,kali kecewa layan hotel mercure kota kali kece...,kali kecewa layan hotel mercure kota kali kece...
2,positif,liburan menyenangkan di jogja stay di hotel yg...,libur senang jogja stay hotel,libur senang jogja stay hotel,libur senang jogja stay hotel
3,positif,lokasinya juga mantap dekat dengan bandara dan...,lokasi mantap bandara wisata kamar nyaman fasi...,lokasi mantap bandara wisata kamar nyaman fasi...,lokasi mantap bandara wisata kamar nyaman fasi...
4,positif,saya merekomendasikan anda untuk menginap di h...,rekomendasi inap hotel layan bagus ramah makan...,rekomendasi inap hotel layan bagus ramah makan...,rekomendasi inap hotel layan bagus ramah makan...


**Extrasi Fitur TF-IDF**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

#  1. Hitung TF-IDF dari lemmatized text
vectorizer = TfidfVectorizer(max_features=500)
tfidf_matrix = vectorizer.fit_transform(df["lemma"].astype(str))
feature_names = vectorizer.get_feature_names_out()

#  2. Ambil 3 kata dengan nilai TF-IDF tertinggi per baris
top_words = []
top_scores = []

for row in tfidf_matrix.toarray():
    idx = np.argsort(row)[::-1][:3]  # 3 kata tertinggi
    words = [(feature_names[i], row[i]) for i in idx if row[i] > 0]

    if words:
        top_words.append(", ".join([w for w, s in words]))
        top_scores.append(", ".join([str(round(s, 4)) for w, s in words]))
    else:
        top_words.append("-")
        top_scores.append("-")

#  3. Buat output tabel ringkas
df_output = pd.DataFrame({
    "Label": df["Label"],
    "Kalimat": df["lemma"],              # kolom teks original bersih
    "Kata_Terkuat_TFIDF": top_words,
    "Skor_TFIDF": top_scores
})

# Tampilkan
print(df_output)

       Label                                            Kalimat  \
0    positif  layan ramah banget fasilitas kamar rekomendasi...   
1    negatif  kali kecewa layan hotel mercure kota kali kece...   
2    positif                      libur senang jogja stay hotel   
3    positif  lokasi mantap bandara wisata kamar nyaman fasi...   
4    positif  rekomendasi inap hotel layan bagus ramah makan...   
..       ...                                                ...   
244  positif  grand mercure jogja pukau hotel indah bersih n...   
245  negatif  hotel lumayan tua pintu kamar mandi kayu bagi ...   
246  negatif  stay hotel accor member gold alam buruk lift m...   
247   netral  layan muas sarap lantai malas orang sarap lant...   
248  negatif  lift pandu guide tamu butuh info masuk halaman...   

                   Kata_Terkuat_TFIDF              Skor_TFIDF  
0          wisata, libur, rekomendasi  0.4888, 0.4356, 0.4356  
1                 kali, kecewa, emosi   0.5339, 0.4655, 0.329  
2  

1.   **☑ Logistic Regression**
2.   **☑ SVM**
3.   **☑ Random Forest**
4.   **☑ Naive Bayes**

In [ ]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score


# STEMMING (penting!)
factory = StemmerFactory()
stemmer = factory.create_stemmer()
df['lemma'] = df['Clean_Text'].apply(lambda x: stemmer.stem(x))

# TF-IDF
X = df['lemma']
y = df['Label']

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)


# SMOTE untuk balancing
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# TRAIN 4 MODEL
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear'),
    "Random Forest": RandomForestClassifier(),
    "Multinomial NB": MultinomialNB()
}

results = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "report": classification_report(y_test, y_pred)
    }

# TAMPILKAN HASIL
for name, res in results.items():
    print("="*70)
    print(f"MODEL: {name}")
    print(f"Akurasi: {res['accuracy']:.4f}")
    print(res['report'])

MODEL: Logistic Regression
Akurasi: 0.8800
              precision    recall  f1-score   support

     negatif       0.88      0.92      0.90        24
      netral       0.33      0.20      0.25         5
     positif       0.95      1.00      0.98        21

    accuracy                           0.88        50
   macro avg       0.72      0.71      0.71        50
weighted avg       0.86      0.88      0.87        50

MODEL: SVM
Akurasi: 0.9000
              precision    recall  f1-score   support

     negatif       0.92      0.92      0.92        24
      netral       0.50      0.40      0.44         5
     positif       0.95      1.00      0.98        21

    accuracy                           0.90        50
   macro avg       0.79      0.77      0.78        50
weighted avg       0.89      0.90      0.89        50

MODEL: Random Forest
Akurasi: 0.8600
              precision    recall  f1-score   support

     negatif       0.85      0.92      0.88        24
      netral       1.0